# NOORIX-AI — Bulletproof Recovery Master Kaggle Notebook (v2)
**Prepared by Muhammad Farhan · NOORIX-AI**

This hardened notebook rebuilds the PKR denomination/side classifier, exports the YOLOv8n COCO model, trains Urdu Whisper ASR, and packages all artifacts into one verified ZIP.

### Key Fixes in v2:
- **Zero-Restart Dependency Management:** Installs only missing/needed packages without breaking Kaggle's pre-installed `pyarrow`, `numpy`, or `torch` binaries, avoiding kernel crash loops.
- **Kaggle `num_workers=0` Safety:** Prevents DataLoader worker deadlocks and shared-memory crashes common in Kaggle containers.
- **Automatic Audio Decoding & Filtering:** Safely skips corrupted/empty WAV files during Whisper preprocessing so a single bad audio clip never aborts training.
- **Stage Isolation (`try/except`):** If one stage fails, the error is logged to `OUT` and remaining stages + final ZIP packaging still complete.
- **Auto-Resume Checkpoints:** Automatically resumes PKR or Whisper training if interrupted within the same session.

**Setup Checklist:**
1. Attach datasets via **Add Input**: `muhammadammarabid/pakistanicurrencydataset` and `muhammadahmedansari/urdu-dataset-20000`.
2. Enable **GPU (T4)** and **Internet** in Notebook Settings.
3. Click **Run All**.

In [1]:
from pathlib import Path
import os, sys, json, re, shutil, zipfile, hashlib, time, traceback, random, inspect, gc
import numpy as np
import pandas as pd
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT = Path("/kaggle/input")
OUT = WORK / "NOORIX_MASTER_OUTPUT"
OUT.mkdir(parents=True, exist_ok=True)
MODELS = OUT / "models"
MODELS.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output dir:", OUT)

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Output dir: /kaggle/working/NOORIX_MASTER_OUTPUT


## 1. Safe Dependency Installation (No Kernel Restart Needed)

Installs `ultralytics`, `onnx`, `evaluate`, `jiwer`, `soundfile`, and `librosa` while preserving Kaggle's pre-installed `torch`, `pyarrow`, and `transformers` ABI compatibility.

In [2]:
import subprocess, importlib

def ensure_pkg(import_name, pip_spec=None):
    pip_spec = pip_spec or import_name
    try:
        importlib.import_module(import_name)
        print(f"[OK] {import_name} already available")
    except ImportError:
        print(f"[INSTALL] {pip_spec}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pip_spec] if "ultralytics" in pip_spec else [sys.executable, "-m", "pip", "install", "-q", pip_spec])

# Install required lightweight packages safely
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "onnx", "onnxslim", "evaluate", "jiwer", "soundfile", "librosa"
])

# Verify core imports immediately
import transformers, datasets, soundfile, librosa, sklearn, onnx
from ultralytics import YOLO
print("Transformers:", transformers.__version__, "| Datasets:", datasets.__version__)
print("All dependencies verified cleanly!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Transformers: 5.20.2 | Datasets: 4.8.5
All dependencies verified cleanly!


## 2. Discover and Validate Datasets

Automatically locates PKR currency image folders anywhere under `/kaggle/input` and validates Urdu TSV/WAV pairs.

In [3]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
pkr_candidates = [
    INPUT / "datasets/muhammadammarabid/pakistanicurrencydataset/data-rescaled",
    INPUT / "pakistanicurrencydataset/data-rescaled",
    INPUT / "datasets/rafayqayyum/pakistan-image-currency-dataset/currencyDataset",
    INPUT / "pakistan-image-currency-dataset/currencyDataset",
]

# Fallback recursive search if mounted at a custom slug
if INPUT.exists():
    for p in INPUT.rglob("data-rescaled"):
        if p.is_dir() and p not in pkr_candidates:
            pkr_candidates.append(p)

pkr_dir, pkr_classes = None, {}
for candidate in pkr_candidates:
    if not candidate.is_dir():
        continue
    summary = {}
    for folder in sorted(candidate.iterdir()):
        if folder.is_dir():
            n = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_exts)
            if n:
                summary[folder.name] = n
    eligible = [
        name for name in summary
        if re.search(r"(?<!\d)(10|20|50|75|100|500|1000|5000)(?!\d)", name)
        and ("front" in name.lower() or "back" in name.lower())
    ]
    if len(eligible) >= 4:
        pkr_dir, pkr_classes = candidate, {k: summary[k] for k in eligible}
        break

print("PKR selected:", pkr_dir)
print("PKR classes:", pkr_classes)
print("PKR total images:", sum(pkr_classes.values()))

urdu_candidates = [
    INPUT / "datasets/muhammadahmedansari/urdu-dataset-20000",
    INPUT / "urdu-dataset-20000",
]
URDU_BASE = next((c for c in urdu_candidates if c.is_dir()), None)
if URDU_BASE is None and INPUT.exists():
    tsv_found = list(INPUT.rglob("final_main_dataset.tsv"))
    if tsv_found:
        URDU_BASE = tsv_found[0].parent

URDU_MANIFEST = OUT / "urdu_asr_manifest.csv"
urdu_count = 0

if URDU_BASE is not None:
    URDU_TSV = URDU_BASE / "final_main_dataset.tsv"
    URDU_AUDIO = URDU_BASE / "limited_wav_files"
    if URDU_TSV.is_file() and URDU_AUDIO.is_dir():
        audio_index = {}
        for p in URDU_AUDIO.rglob("*.wav"):
            if p.is_file() and p.stat().st_size > 1000:  # Ignore empty/corrupt 0-byte WAVs
                audio_index.setdefault(p.stem, str(p))
        speech = pd.read_csv(URDU_TSV, sep="\t")
        required = {"path", "sentence", "locale"}
        if required.issubset(speech.columns):
            speech["audio_path"] = speech["path"].astype(str).map(lambda x: audio_index.get(Path(x).stem))
            speech["text"] = speech["sentence"].fillna("").astype(str).str.strip()
            speech = speech[
                speech["audio_path"].notna()
                & speech["text"].ne("")
                & speech["locale"].astype(str).eq("ur")
            ].copy()
            if "client_id" not in speech.columns:
                speech["client_id"] = "unknown"
            speech = speech.drop_duplicates(subset=["audio_path"])
            speech[["audio_path", "text", "client_id"]].to_csv(URDU_MANIFEST, index=False)
            urdu_count = len(speech)

print("Urdu valid audio/transcript pairs:", urdu_count)

PKR selected: /kaggle/input/datasets/muhammadammarabid/pakistanicurrencydataset/data-rescaled
PKR classes: {'1000_back': 236, '1000_front': 227, '100_back': 281, '100_front': 280, '10_back': 279, '10_front': 268, '20_back': 258, '20_front': 248, '5000_back': 261, '5000_front': 257, '500_back': 261, '500_front': 262, '50_back': 253, '50_front': 240}
PKR total images: 3611
Urdu valid audio/transcript pairs: 15492


## 3. Training Configuration

In [4]:
RUN_PKR = pkr_dir is not None
RUN_YOLO = True
RUN_URDU_ASR = urdu_count >= 100

PKR_EPOCHS = 5
PKR_BATCH_SIZE = 32

ASR_MAX_SAMPLES = 5000
ASR_EPOCHS = 2
ASR_BATCH_SIZE = 2
ASR_GRAD_ACCUM = 4
ASR_SAVE_STEPS = 250

print(f"Stages enabled -> PKR: {RUN_PKR} | YOLO: {RUN_YOLO} | Urdu ASR: {RUN_URDU_ASR}")

Stages enabled -> PKR: True | YOLO: True | Urdu ASR: True


## 4. Train PKR Denomination/Side Classifier

Trains MobileNetV2 with `num_workers=0` (preventing Kaggle multiprocessing freezes), mixed-precision AMP on GPU, and checkpoint resumption.

In [5]:
pkr_status = "skipped"
if RUN_PKR:
    try:
        from torchvision import datasets, transforms, models
        from torch.utils.data import DataLoader
        import torch.nn as nn
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import accuracy_score

        pkr_out = MODELS / "pkr_currency"
        pkr_out.mkdir(parents=True, exist_ok=True)

        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomRotation(8),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        base_ds = datasets.ImageFolder(str(pkr_dir))
        keep_classes = [
            c for c in base_ds.classes
            if re.search(r"(?<!\d)(10|20|50|75|100|500|1000|5000)(?!\d)", c)
            and ("front" in c.lower() or "back" in c.lower())
        ]
        old_to_new = {base_ds.class_to_idx[c]: i for i, c in enumerate(keep_classes)}
        samples = [(path, old_to_new[y]) for path, y in base_ds.samples if y in old_to_new]
        targets = [y for _, y in samples]

        tr_idx, va_idx = train_test_split(
            np.arange(len(samples)), test_size=0.2, random_state=SEED, stratify=targets
        )

        class SafePKRDataset(torch.utils.data.Dataset):
            def __init__(self, items, tfm):
                self.items = items
                self.tfm = tfm
                self.loader = base_ds.loader
            def __len__(self):
                return len(self.items)
            def __getitem__(self, idx):
                path, label = self.items[idx]
                img = self.loader(path)
                return self.tfm(img), label

        device = "cuda" if torch.cuda.is_available() else "cpu"
        # num_workers=0 avoids Kaggle /dev/shm multiprocessing deadlocks
        train_loader = DataLoader(SafePKRDataset([samples[i] for i in tr_idx], train_transform), batch_size=PKR_BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(SafePKRDataset([samples[i] for i in va_idx], val_transform), batch_size=PKR_BATCH_SIZE, shuffle=False, num_workers=0)

        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.last_channel, len(keep_classes))
        model = model.to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()

        start_epoch = 0
        best_acc = -1.0
        ckpt_file = pkr_out / "latest_checkpoint.pth"
        best_file = pkr_out / "mobilenetv2_pkr_best.pth"

        if ckpt_file.exists():
            ckpt = torch.load(ckpt_file, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            start_epoch = ckpt.get("epoch", 0)
            best_acc = ckpt.get("best_accuracy", -1.0)
            print(f"Resuming PKR training from epoch {start_epoch}")

        for epoch in range(start_epoch, PKR_EPOCHS):
            model.train()
            running = []
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()
                running.append(loss.item())

            model.eval()
            truth, pred = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    logits = model(x.to(device))
                    pred.extend(logits.argmax(1).cpu().tolist())
                    truth.extend(y.tolist())
            acc = accuracy_score(truth, pred)
            print(f"PKR Epoch {epoch+1}/{PKR_EPOCHS} | loss={np.mean(running):.4f} | val_acc={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), best_file)
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_accuracy": best_acc,
                "classes": keep_classes
            }, ckpt_file)

        (pkr_out / "labels.json").write_text(json.dumps({"classes": keep_classes}, ensure_ascii=False, indent=2), encoding="utf-8")
        (pkr_out / "metadata.json").write_text(json.dumps({
            "architecture": "MobileNetV2",
            "validation_accuracy": float(best_acc),
            "train_samples": len(tr_idx),
            "validation_samples": len(va_idx),
            "note": "Denomination/side classification only; not authenticity detection."
        }, indent=2), encoding="utf-8")

        del model, optimizer, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        pkr_status = "success"
    except Exception as e:
        pkr_status = f"failed: {type(e).__name__}"
        print("PKR stage error:", repr(e))
        (OUT / "pkr_currency_ERROR.txt").write_text(traceback.format_exc(), encoding="utf-8")

print("PKR status:", pkr_status)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 124MB/s]


PKR Epoch 1/5 | loss=1.5713 | val_acc=0.9917
PKR Epoch 2/5 | loss=0.1230 | val_acc=0.9945
PKR Epoch 3/5 | loss=0.0428 | val_acc=0.9959
PKR Epoch 4/5 | loss=0.0223 | val_acc=0.9972
PKR Epoch 5/5 | loss=0.0153 | val_acc=0.9945
PKR status: success


## 5. Export YOLOv8n Object/Person Detector

In [6]:
yolo_status = "skipped"
if RUN_YOLO:
    try:
        from ultralytics import YOLO
        yolo_out = MODELS / "yolo_objects_person"
        yolo_out.mkdir(parents=True, exist_ok=True)
        yolo = YOLO("yolov8n.pt")
        shutil.copy2("yolov8n.pt", yolo_out / "yolov8n.pt")
        exported = yolo.export(format="onnx", imgsz=640, dynamic=False, simplify=False)
        shutil.copy2(exported, yolo_out / Path(exported).name)
        (yolo_out / "labels.json").write_text(
            json.dumps({str(k): v for k, v in yolo.names.items()}, indent=2),
            encoding="utf-8"
        )
        (yolo_out / "metadata.json").write_text(json.dumps({
            "model": "YOLOv8n",
            "source": "COCO pretrained weights",
            "person_class_included": True,
            "note": "Requires a YOLO-specific decoder; not plug-compatible with an SSD output adapter."
        }, indent=2), encoding="utf-8")
        del yolo
        gc.collect()
        yolo_status = "success_pretrained_export"
    except Exception as e:
        yolo_status = f"failed: {type(e).__name__}"
        print("YOLO export failed:", repr(e))
        (OUT / "yolo_objects_person_ERROR.txt").write_text(traceback.format_exc(), encoding="utf-8")

print("YOLO status:", yolo_status)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: export success ✅ 0.7s, saved as 'yolov8n.onnx' (12.2 MB)

Export complete (1.2s)
Results saved to /kaggle/working/yolov8n.onnx
Predict:         yolo predict task=detect model=yolov8n.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n.onnx imgsz=640 data=coco.yaml 
Visualize:       https://netron.app
YOLO status: success_pretrained_export


## 6. Prepare & Train Urdu Whisper ASR

Uses `soundfile`/`librosa` directly (bypassing `torchcodec`/FFmpeg ABI issues in newer Python versions), filters out broken WAV files safely, and trains with `dataloader_num_workers=0`.

In [7]:
asr_status = "skipped"
if RUN_URDU_ASR:
    try:
        import librosa
        from dataclasses import dataclass
        from datasets import Dataset
        from sklearn.model_selection import GroupShuffleSplit, train_test_split
        from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments

        df = pd.read_csv(URDU_MANIFEST).dropna(subset=["audio_path", "text"]).copy()
        df["text"] = df["text"].astype(str).str.strip()
        df = df[df["text"].ne("")]
        if len(df) > ASR_MAX_SAMPLES:
            df = df.sample(ASR_MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

        if "client_id" in df.columns and df["client_id"].nunique() >= 2:
            splitter = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
            train_idx, val_idx = next(splitter.split(df, groups=df["client_id"].fillna("unknown")))
            train_df = df.iloc[train_idx].drop(columns=["client_id"], errors="ignore").reset_index(drop=True)
            val_df = df.iloc[val_idx].drop(columns=["client_id"], errors="ignore").reset_index(drop=True)
        else:
            train_df, val_df = train_test_split(df.drop(columns=["client_id"], errors="ignore"), test_size=0.1, random_state=SEED)
            train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

        MODEL_NAME = "openai/whisper-small"
        LANGUAGE = "urdu"
        processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task="transcribe")
        model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
        model.generation_config.language = LANGUAGE
        model.generation_config.task = "transcribe"
        model.generation_config.forced_decoder_ids = None
        model.config.use_cache = False

        def prepare_safe(batch):
            try:
                speech_arr, _ = librosa.load(batch["audio_path"], sr=16000, mono=True)
                if speech_arr is None or len(speech_arr) < 1600:  # < 0.1s
                    return {"input_features": None, "labels": None, "valid": False}
                feats = processor.feature_extractor(speech_arr, sampling_rate=16000).input_features[0]
                lbls = processor.tokenizer(batch["text"]).input_ids
                if len(lbls) == 0 or len(lbls) > 448:
                    return {"input_features": None, "labels": None, "valid": False}
                return {"input_features": feats, "labels": lbls, "valid": True}
            except Exception:
                return {"input_features": None, "labels": None, "valid": False}

        print("Extracting audio features safely...")
        train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(prepare_safe, remove_columns=train_df.columns.tolist())
        val_ds = Dataset.from_pandas(val_df, preserve_index=False).map(prepare_safe, remove_columns=val_df.columns.tolist())
        train_ds = train_ds.filter(lambda x: x["valid"]).remove_columns(["valid"])
        val_ds = val_ds.filter(lambda x: x["valid"]).remove_columns(["valid"])
        print(f"Valid samples -> Train: {len(train_ds)} | Val: {len(val_ds)}")

        @dataclass
        class WhisperDataCollator:
            processor: object
            def __call__(self, features):
                inputs = [{"input_features": f["input_features"]} for f in features]
                batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")
                labels = [{"input_ids": f["labels"]} for f in features]
                label_batch = self.processor.tokenizer.pad(labels, return_tensors="pt")
                batch["labels"] = label_batch["input_ids"].masked_fill(label_batch["attention_mask"].ne(1), -100)
                return batch

        asr_out = MODELS / "whisper_urdu_asr"
        asr_out.mkdir(parents=True, exist_ok=True)
        checkpoint_dir = WORK / "noorix_urdu_asr_checkpoints"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        args_dict = dict(
            output_dir=str(checkpoint_dir),
            per_device_train_batch_size=ASR_BATCH_SIZE,
            per_device_eval_batch_size=ASR_BATCH_SIZE,
            gradient_accumulation_steps=ASR_GRAD_ACCUM,
            learning_rate=1e-5,
            num_train_epochs=ASR_EPOCHS,
            fp16=torch.cuda.is_available(),
            save_strategy="steps",
            save_steps=ASR_SAVE_STEPS,
            save_total_limit=2,
            eval_steps=ASR_SAVE_STEPS,
            logging_steps=50,
            report_to="none",
            dataloader_num_workers=0,
            gradient_checkpointing=True,
            predict_with_generate=False
        )
        sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
        args_dict["eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"] = "steps"

        training_args = Seq2SeqTrainingArguments(**args_dict)
        trainer_kwargs = dict(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=WhisperDataCollator(processor)
        )
        trainer_sig = inspect.signature(Seq2SeqTrainer.__init__)
        trainer_kwargs["processing_class" if "processing_class" in trainer_sig.parameters else "tokenizer"] = processor

        trainer = Seq2SeqTrainer(**trainer_kwargs)
        checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        resume_path = str(checkpoints[-1]) if checkpoints else None
        print("Whisper resume checkpoint:", resume_path or "none (starting fresh)")

        trainer.train(resume_from_checkpoint=resume_path)
        metrics = trainer.evaluate()
        trainer.save_model(str(asr_out / "model"))
        processor.save_pretrained(asr_out / "processor")
        (asr_out / "metadata.json").write_text(json.dumps({
            "task": "Urdu speech-to-text (ASR)",
            "base_model": MODEL_NAME,
            "train_samples": len(train_ds),
            "validation_samples": len(val_ds),
            "epochs_requested": ASR_EPOCHS,
            "evaluation_metrics": {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))},
            "note": "ASR is not TTS. Validate on a separate real-world test set."
        }, ensure_ascii=False, indent=2), encoding="utf-8")

        del trainer, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        asr_status = "success"
    except Exception as e:
        asr_status = f"failed: {type(e).__name__}"
        print("Urdu ASR stage error:", repr(e))
        (OUT / "whisper_urdu_asr_ERROR.txt").write_text(traceback.format_exc(), encoding="utf-8")

print("Urdu ASR status:", asr_status)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (21 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step
16.414438,2.464350,564


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Urdu ASR status: success


## 7. Package All Outputs Into Verified ZIP

In [8]:
stage_records = [
    {"stage": "pkr_currency", "status": pkr_status},
    {"stage": "yolo_objects_person", "status": yolo_status},
    {"stage": "whisper_urdu_asr", "status": asr_status}
]
manifest = {
    "project": "NOORIX-AI",
    "prepared_by": "Muhammad Farhan",
    "package_version": "recovery-master-v2-bulletproof",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "architecture": "Unified application/API with task-specific specialist models",
    "stages": stage_records,
    "warnings": [
        "PKR model predicts denomination/side only; it does not authenticate notes.",
        "YOLOv8n is a pretrained COCO export unless custom labelled training is separately configured.",
        "YOLO ONNX needs a YOLO-specific decoder; it is not compatible with the existing SSD adapter by default.",
        "Whisper Urdu is speech-to-text, not text-to-speech.",
        "OCR, English ASR, and TTS are not trained by this notebook.",
        "Model export does not prove desktop/mobile integration."
    ]
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "README_OFFLINE.md").write_text(
    "# NOORIX-AI model package\n\n"
    "See manifest.json for each stage status. Install compatible runtimes and adapters before offline inference. "
    "YOLO requires its own decoder. PKR classification is not authenticity detection. Whisper is ASR, not TTS.\n",
    encoding="utf-8"
)

checksum_lines = []
for f in sorted(p for p in OUT.rglob("*") if p.is_file() and p.name != "checksums.sha256"):
    checksum_lines.append(hashlib.sha256(f.read_bytes()).hexdigest() + "  " + f.relative_to(OUT).as_posix())
(OUT / "checksums.sha256").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

zip_path = WORK / "NOORIX_AI_ALL_MODELS.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(p for p in OUT.rglob("*") if p.is_file()):
        zf.write(f, arcname=f.relative_to(OUT).as_posix())

with zipfile.ZipFile(zip_path, "r") as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError("ZIP integrity check failed: " + str(bad))

print("FINAL ZIP:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))
print("ZIP entries:", len(zipfile.ZipFile(zip_path).namelist()))
print(json.dumps(manifest, indent=2, ensure_ascii=False))

FINAL ZIP: /kaggle/working/NOORIX_AI_ALL_MODELS.zip
Size MB: 904.64
ZIP entries: 23
{
  "project": "NOORIX-AI",
  "prepared_by": "Muhammad Farhan",
  "package_version": "recovery-master-v2-bulletproof",
  "created_utc": "2026-11-16T14:13:43Z",
  "architecture": "Unified application/API with task-specific specialist models",
  "stages": [
    {
      "stage": "pkr_currency",
      "status": "success"
    },
    {
      "stage": "yolo_objects_person",
      "status": "success_pretrained_export"
    },
    {
      "stage": "whisper_urdu_asr",
      "status": "success"
    }
  ],
  "warnings": [
    "PKR model predicts denomination/side only; it does not authenticate notes.",
    "YOLOv8n is a pretrained COCO export unless custom labelled training is separately configured.",
    "YOLO ONNX needs a YOLO-specific decoder; it is not compatible with the existing SSD adapter by default.",
    "Whisper Urdu is speech-to-text, not text-to-speech.",
    "OCR, English ASR, and TTS are not trained b